<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/DL-2026/Lecture_2/Lecture_2_Multilayer_Perceptron_and_Backpropagation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Многослойный перцептрон и алгоритм обратного распространения ошибки: подробное изложение

### 1. Введение в многослойный перцептрон

Многослойный перцептрон (Multilayer Perceptron, MLP) представляет собой класс искусственных нейронных сетей прямого распространения, в которых нейроны организованы в последовательные слои: входной слой, один или несколько скрытых слоёв и выходной слой. Каждый нейрон последующего слоя принимает выходы всех нейронов предыдущего слоя, вычисляет их взвешенную сумму с добавлением смещения (bias) и применяет нелинейную функцию активации. Благодаря наличию хотя бы одного скрытого слоя с нелинейной активацией MLP способен аппроксимировать произвольные непрерывные функции (теорема Цыбенко), что делает его универсальным инструментом для задач регрессии, классификации и многих других.

Обучение MLP состоит в настройке весовых коэффициентов и смещений таким образом, чтобы минимизировать заранее выбранную функцию потерь, измеряющую расхождение между предсказаниями сети и истинными целевыми значениями. Наиболее распространённым методом оптимизации является градиентный спуск, который требует вычисления градиентов функции потерь по всем параметрам сети. Для эффективного вычисления этих градиентов используется алгоритм **обратного распространения ошибки** (error backpropagation), основанный на последовательном применении цепного правила дифференцирования.

В настоящем разделе мы рассмотрим три конкретные архитектуры MLP возрастающей сложности, детально выведем все формулы прямого прохода (forward pass) и обратного распространения (backward pass), введём удобное обозначение локального градиента $\delta$ и объясним его роль в упрощении вычислений. Для определённости будем использовать сигмоидную функцию активации  
$$
\sigma(z) = \frac{1}{1+e^{-z}}, \qquad \sigma'(z) = \sigma(z)\bigl(1-\sigma(z)\bigr),
$$
и квадратичную функцию потерь  
$$
L = \frac12 \sum_{i} (y_i - \hat y_i)^2,
$$
где $y_i$ — истинные значения, $\hat y_i$ — предсказания сети. Множитель $1/2$ введён для упрощения производной.

---

### 2. Основные понятия и обозначения

#### 2.1. Послойное представление

Рассмотрим сеть с $L$ слоями (включая входной). Пронумеруем слои от $0$ до $L$, где слой $0$ — входной, слой $L$ — выходной. Для простоты мы ограничимся случаем $L=2$: входной слой (слой 0), один скрытый слой (слой 1) и выходной слой (слой 2). Однако все выкладки непосредственно обобщаются на произвольное число слоёв.

Каждый слой $\ell$ содержит $n_\ell$ нейронов. Активации нейронов слоя $\ell$ обозначим вектором $\mathbf a^{(\ell)} = (a_1^{(\ell)}, \dots, a_{n_\ell}^{(\ell)})^\top$. Входной вектор $\mathbf x = (x_1, \dots, x_{n_0})^\top$ можно рассматривать как активации слоя 0: $\mathbf a^{(0)} = \mathbf x$.

Связь между слоями $\ell-1$ и $\ell$ задаётся матрицей весов $\mathbf W^{(\ell)} \in \mathbb R^{n_\ell \times n_{\ell-1}}$, где элемент $w_{ij}^{(\ell)}$ — вес от нейрона $j$ предыдущего слоя к нейрону $i$ текущего слоя, и вектором смещений $\mathbf b^{(\ell)} \in \mathbb R^{n_\ell}$. Линейная комбинация входов нейрона $i$ слоя $\ell$ обозначается $z_i^{(\ell)}$ и вычисляется как  
$$
z_i^{(\ell)} = \sum_{j=1}^{n_{\ell-1}} w_{ij}^{(\ell)} a_j^{(\ell-1)} + b_i^{(\ell)}.
$$
В векторной форме  
$$
\mathbf z^{(\ell)} = \mathbf W^{(\ell)} \mathbf a^{(\ell-1)} + \mathbf b^{(\ell)}.
$$
Затем применяется функция активации (поэлементно):  
$$
\mathbf a^{(\ell)} = \sigma(\mathbf z^{(\ell)}).
$$
Выход сети $\hat{\mathbf y}$ равен активациям последнего слоя: $\hat{\mathbf y} = \mathbf a^{(L)}$.

#### 2.2. Функция потерь и цель обучения

Пусть $\mathbf y = (y_1,\dots, y_{n_L})^\top$ — вектор целевых значений. Определим квадратичную функцию потерь  
$$
L = \frac12 \|\mathbf y - \hat{\mathbf y}\|^2 = \frac12 \sum_{i=1}^{n_L} (y_i - \hat y_i)^2.
$$
Цель обучения — найти такие параметры $\mathbf W^{(\ell)}, \mathbf b^{(\ell)}$ для всех $\ell$, которые минимизируют $L$. Для этого используется градиентный спуск: на каждой итерации параметры обновляются в направлении, противоположном градиенту:  
$$
\mathbf W^{(\ell)} := \mathbf W^{(\ell)} - \eta \frac{\partial L}{\partial \mathbf W^{(\ell)}}, \qquad
\mathbf b^{(\ell)} := \mathbf b^{(\ell)} - \eta \frac{\partial L}{\partial \mathbf b^{(\ell)}},
$$
где $\eta > 0$ — скорость обучения. Таким образом, необходимо уметь вычислять частные производные $L$ по всем $w_{ij}^{(\ell)}$ и $b_i^{(\ell)}$.

#### 2.3. Цепное правило и локальные градиенты

Функция потерь $L$ является сложной функцией параметров сети, так как зависит от них через последовательность линейных комбинаций и активаций. Для вычисления производных применяется правило дифференцирования сложной функции. Ключевым промежуточным объектом является **локальный градиент** (или ошибка) слоя $\ell$:  
$$
\delta_i^{(\ell)} := \frac{\partial L}{\partial z_i^{(\ell)}}, \quad i=1,\dots,n_\ell.
$$
Вектор $\boldsymbol\delta^{(\ell)} = (\delta_1^{(\ell)}, \dots, \delta_{n_\ell}^{(\ell)})^\top$ показывает, насколько изменение линейной комбинации каждого нейрона слоя $\ell$ влияет на функцию потерь. Зная $\boldsymbol\delta^{(\ell)}$, легко вычислить градиенты по параметрам этого слоя, поскольку  
$$
\frac{\partial z_i^{(\ell)}}{\partial w_{ij}^{(\ell)}} = a_j^{(\ell-1)}, \qquad
\frac{\partial z_i^{(\ell)}}{\partial b_i^{(\ell)}} = 1,
$$
откуда  
$$
\frac{\partial L}{\partial w_{ij}^{(\ell)}} = \delta_i^{(\ell)} \, a_j^{(\ell-1)}, \qquad
\frac{\partial L}{\partial b_i^{(\ell)}} = \delta_i^{(\ell)}.
$$
Таким образом, задача сводится к вычислению $\boldsymbol\delta^{(\ell)}$ для всех слоёв, начиная с выходного и двигаясь к входному. Именно эта рекурсивная процедура составляет суть алгоритма обратного распространения.

---

### 3. Пример 1: один вход, один скрытый нейрон, один выход

Начнём с простейшей архитектуры, позволяющей наглядно проследить все этапы вывода. Сеть имеет один входной нейрон, один скрытый нейрон и один выходной нейрон.

#### 3.1. Архитектура и параметры

Обозначим вход $x$. Скрытый нейрон имеет вес $w_1$ и смещение $b_1$, выходной нейрон — вес $w_2$ и смещение $b_2$. Всего четыре параметра.

#### 3.2. Прямой проход

Прямой проход вычисляет выход сети для заданного входа $x$:

1. Линейная комбинация скрытого нейрона:  
   $$
   z_1 = w_1 x + b_1.
   $$
2. Активация скрытого нейрона:  
   $$
   a_1 = \sigma(z_1).
   $$
3. Линейная комбинация выходного нейрона:  
   $$
   z_2 = w_2 a_1 + b_2.
   $$
4. Выход сети:  
   $$
   \hat y = \sigma(z_2).
   $$

#### 3.3. Обратный проход

Функция потерь: $L = \frac12 (\hat y - y)^2$. Вычислим градиенты по всем параметрам с помощью цепного правила.

**Шаг 1. Производная по выходу сети.**  
$$
\frac{\partial L}{\partial \hat y} = \hat y - y.
$$

**Шаг 2. Производная выхода по линейной комбинации $z_2$.**  
$$
\frac{\partial \hat y}{\partial z_2} = \sigma'(z_2).
$$

**Шаг 3. Локальный градиент выходного слоя.**  
Вводим обозначение  
$$
\delta_2 := \frac{\partial L}{\partial z_2}
= \frac{\partial L}{\partial \hat y} \cdot \frac{\partial \hat y}{\partial z_2}
= (\hat y - y)\,\sigma'(z_2).
$$

**Шаг 4. Градиенты параметров выходного слоя.**  
Используя $\delta_2$, сразу получаем:  
$$
\frac{\partial L}{\partial w_2} = \delta_2 \cdot \frac{\partial z_2}{\partial w_2} = \delta_2 \cdot a_1, \qquad
\frac{\partial L}{\partial b_2} = \delta_2 \cdot \frac{\partial z_2}{\partial b_2} = \delta_2 \cdot 1 = \delta_2.
$$

**Шаг 5. Переход к скрытому слою.**  
Нам нужно вычислить $\delta_1 := \frac{\partial L}{\partial z_1}$. По цепному правилу, учитывая, что $z_2$ зависит от $a_1$, а $a_1$ от $z_1$:  
$$
\delta_1 = \frac{\partial L}{\partial z_1}
= \frac{\partial L}{\partial z_2} \cdot \frac{\partial z_2}{\partial a_1} \cdot \frac{\partial a_1}{\partial z_1}.
$$
Найдём промежуточные производные:  
$$
\frac{\partial z_2}{\partial a_1} = w_2, \qquad
\frac{\partial a_1}{\partial z_1} = \sigma'(z_1).
$$
Тогда  
$$
\delta_1 = \delta_2 \, w_2 \, \sigma'(z_1).
$$

**Шаг 6. Градиенты параметров скрытого слоя.**  
$$
\frac{\partial L}{\partial w_1} = \delta_1 \cdot \frac{\partial z_1}{\partial w_1} = \delta_1 \cdot x, \qquad
\frac{\partial L}{\partial b_1} = \delta_1 \cdot \frac{\partial z_1}{\partial b_1} = \delta_1.
$$

#### 3.4. Обновление параметров

Все параметры обновляются по правилу градиентного спуска:  
$$
w_1 := w_1 - \eta \frac{\partial L}{\partial w_1}, \quad
b_1 := b_1 - \eta \frac{\partial L}{\partial b_1}, \quad
w_2 := w_2 - \eta \frac{\partial L}{\partial w_2}, \quad
b_2 := b_2 - \eta \frac{\partial L}{\partial b_2}.
$$

---

### 4. Пример 2: один вход, два скрытых нейрона, один выход

Теперь увеличим число скрытых нейронов до двух. Это позволит понять, как распространяются ошибки при нескольких путях от входа к выходу.

#### 4.1. Архитектура и параметры

Вход $x$. Скрытый слой содержит два нейрона с параметрами:  
- нейрон 1: вес $w_{11}$, смещение $b_{11}$;  
- нейрон 2: вес $w_{12}$, смещение $b_{12}$.

Выходной нейрон имеет два веса $w_{21}, w_{22}$ (от первого и второго скрытых нейронов соответственно) и одно смещение $b_2$. Итого 7 параметров.

#### 4.2. Прямой проход

1. Линейные комбинации скрытого слоя:  
   $$
   z_{11} = w_{11} x + b_{11}, \qquad z_{12} = w_{12} x + b_{12}.
   $$
2. Активации скрытого слоя:  
   $$
   h_1 = \sigma(z_{11}), \qquad h_2 = \sigma(z_{12}).
   $$
   (Мы используем обозначения $h_1,h_2$ для активаций скрытого слоя, чтобы отличать их от $a_1$ в предыдущем примере; в общем случае это $a^{(1)}_1, a^{(1)}_2$.)
3. Линейная комбинация выходного нейрона:  
   $$
   z_2 = w_{21} h_1 + w_{22} h_2 + b_2.
   $$
4. Выход:  
   $$
   \hat y = \sigma(z_2).
   $$

#### 4.3. Обратный проход

Функция потерь $L = \frac12(\hat y - y)^2$.

**Шаг 1.** $\frac{\partial L}{\partial \hat y} = \hat y - y$.

**Шаг 2.** $\frac{\partial \hat y}{\partial z_2} = \sigma'(z_2)$.

**Шаг 3.** $\delta_2 := \frac{\partial L}{\partial z_2} = (\hat y - y)\,\sigma'(z_2)$.

**Шаг 4.** Градиенты выходного слоя:  
$$
\frac{\partial L}{\partial w_{21}} = \delta_2 \cdot h_1, \quad
\frac{\partial L}{\partial w_{22}} = \delta_2 \cdot h_2, \quad
\frac{\partial L}{\partial b_2} = \delta_2.
$$

**Шаг 5.** Переход к скрытому слою. Нам нужно вычислить $\delta_{11} = \frac{\partial L}{\partial z_{11}}$ и $\delta_{12} = \frac{\partial L}{\partial z_{12}}$.

Рассмотрим первый скрытый нейрон. Его линейная комбинация $z_{11}$ влияет на выход только через активацию $h_1$, которая входит в $z_2$ с весом $w_{21}$. Цепное правило даёт:  
$$
\delta_{11} = \frac{\partial L}{\partial z_{11}}
= \frac{\partial L}{\partial z_2} \cdot \frac{\partial z_2}{\partial h_1} \cdot \frac{\partial h_1}{\partial z_{11}}
= \delta_2 \cdot w_{21} \cdot \sigma'(z_{11}).
$$
Аналогично для второго нейрона:  
$$
\delta_{12} = \delta_2 \cdot w_{22} \cdot \sigma'(z_{12}).
$$

**Шаг 6.** Градиенты скрытого слоя.  
Для каждого скрытого нейрона $k=1,2$ имеем:  
$$
\frac{\partial L}{\partial w_{1k}} = \delta_{1k} \cdot x, \qquad
\frac{\partial L}{\partial b_{1k}} = \delta_{1k}.
$$
(Здесь нет суммирования, так как каждый скрытый нейрон соединён только с одним выходным нейроном.)

#### 4.4. Обновление параметров

Все параметры обновляются по правилу $\theta := \theta - \eta \partial L/\partial \theta$.

---

### 5. Пример 3: два входа, два скрытых нейрона, два выхода

Это наиболее общий из рассматриваемых примеров. Здесь появляется суммирование при обратном распространении, поскольку каждый скрытый нейрон влияет на оба выходных нейрона.

#### 5.1. Архитектура и параметры

Входной вектор $\mathbf x = (x_1,x_2)^\top$. Скрытый слой из двух нейронов, выходной слой из двух нейронов. Для единообразия будем использовать матричные обозначения.

Параметры:  
- $\mathbf W^{(1)} \in \mathbb R^{2\times 2}$ — матрица весов между входным и скрытым слоями,  
  $$
  \mathbf W^{(1)} = \begin{pmatrix}
  w_{11}^{(1)} & w_{12}^{(1)} \\
  w_{21}^{(1)} & w_{22}^{(1)}
  \end{pmatrix},
  $$
  где $w_{ij}^{(1)}$ — вес от $j$-го входа к $i$-му скрытому нейрону.  
- $\mathbf b^{(1)} = (b_1^{(1)}, b_2^{(1)})^\top$ — смещения скрытого слоя.  
- $\mathbf W^{(2)} \in \mathbb R^{2\times 2}$ — матрица весов между скрытым и выходным слоями,  
  $$
  \mathbf W^{(2)} = \begin{pmatrix}
  w_{11}^{(2)} & w_{12}^{(2)} \\
  w_{21}^{(2)} & w_{22}^{(2)}
  \end{pmatrix},
  $$
  где $w_{ij}^{(2)}$ — вес от $j$-го скрытого нейрона к $i$-му выходному.  
- $\mathbf b^{(2)} = (b_1^{(2)}, b_2^{(2)})^\top$ — смещения выходного слоя.

#### 5.2. Прямой проход

В матричной форме:  
$$
\mathbf z^{(1)} = \mathbf W^{(1)} \mathbf x + \mathbf b^{(1)}, \qquad
\mathbf h = \sigma(\mathbf z^{(1)}),
$$
$$
\mathbf z^{(2)} = \mathbf W^{(2)} \mathbf h + \mathbf b^{(2)}, \qquad
\hat{\mathbf y} = \sigma(\mathbf z^{(2)}).
$$
В скалярной форме (для наглядности):  
$$
z_1^{(1)} = w_{11}^{(1)} x_1 + w_{12}^{(1)} x_2 + b_1^{(1)}, \quad h_1 = \sigma(z_1^{(1)}),
$$
$$
z_2^{(1)} = w_{21}^{(1)} x_1 + w_{22}^{(1)} x_2 + b_2^{(1)}, \quad h_2 = \sigma(z_2^{(1)}),
$$
$$
z_1^{(2)} = w_{11}^{(2)} h_1 + w_{12}^{(2)} h_2 + b_1^{(2)}, \quad \hat y_1 = \sigma(z_1^{(2)}),
$$
$$
z_2^{(2)} = w_{21}^{(2)} h_1 + w_{22}^{(2)} h_2 + b_2^{(2)}, \quad \hat y_2 = \sigma(z_2^{(2)}).
$$

#### 5.3. Функция потерь

$$
L = \frac12 \left[ (\hat y_1 - y_1)^2 + (\hat y_2 - y_2)^2 \right].
$$

#### 5.4. Обратный проход

**Шаг 1. Градиенты по выходам.**  
$$
\frac{\partial L}{\partial \hat y_1} = \hat y_1 - y_1, \qquad
\frac{\partial L}{\partial \hat y_2} = \hat y_2 - y_2.
$$

**Шаг 2. Производные выходов по своим линейным комбинациям.**  
$$
\frac{\partial \hat y_1}{\partial z_1^{(2)}} = \sigma'(z_1^{(2)}), \qquad
\frac{\partial \hat y_2}{\partial z_2^{(2)}} = \sigma'(z_2^{(2)}).
$$

**Шаг 3. Локальные градиенты выходного слоя.**  
$$
\delta_1^{(2)} := \frac{\partial L}{\partial z_1^{(2)}} = (\hat y_1 - y_1)\,\sigma'(z_1^{(2)}),
$$
$$
\delta_2^{(2)} := \frac{\partial L}{\partial z_2^{(2)}} = (\hat y_2 - y_2)\,\sigma'(z_2^{(2)}).
$$

**Шаг 4. Градиенты параметров выходного слоя.**  
Для каждого $i=1,2$ и $j=1,2$:  
$$
\frac{\partial L}{\partial w_{ij}^{(2)}} = \delta_i^{(2)} \cdot h_j, \qquad
\frac{\partial L}{\partial b_i^{(2)}} = \delta_i^{(2)}.
$$

**Шаг 5. Переход к скрытому слою: вычисление $\delta_j^{(1)}$.**

Теперь нам нужно вычислить $\delta_1^{(1)} = \frac{\partial L}{\partial z_1^{(1)}}$ и $\delta_2^{(1)} = \frac{\partial L}{\partial z_2^{(1)}}$. Здесь возникает принципиальное отличие от предыдущих примеров: каждый скрытый нейрон $j$ влияет на оба выходных нейрона, а значит, на оба слагаемых функции потерь. Поэтому полная производная $L$ по $z_j^{(1)}$ должна учитывать оба пути: через выход 1 и через выход 2.

Рассмотрим, например, $\delta_1^{(1)}$. Линейная комбинация $z_1^{(1)}$ определяет активацию $h_1 = \sigma(z_1^{(1)})$. Активация $h_1$ входит в обе линейные комбинации выходного слоя:  
$$
z_1^{(2)} = w_{11}^{(2)} h_1 + w_{12}^{(2)} h_2 + b_1^{(2)}, \qquad
z_2^{(2)} = w_{21}^{(2)} h_1 + w_{22}^{(2)} h_2 + b_2^{(2)}.
$$
Следовательно, изменение $z_1^{(1)}$ (а значит и $h_1$) повлияет и на $\hat y_1$, и на $\hat y_2$, а через них на $L$. По правилу дифференцирования сложной функции для многих переменных полная производная равна сумме частных вкладов:  
$$
\delta_1^{(1)} = \frac{\partial L}{\partial z_1^{(1)}}
= \frac{\partial L}{\partial z_1^{(2)}} \cdot \frac{\partial z_1^{(2)}}{\partial h_1} \cdot \frac{\partial h_1}{\partial z_1^{(1)}}
+ \frac{\partial L}{\partial z_2^{(2)}} \cdot \frac{\partial z_2^{(2)}}{\partial h_1} \cdot \frac{\partial h_1}{\partial z_1^{(1)}}.
$$
Подставляя известные величины:  
$$
\frac{\partial L}{\partial z_1^{(2)}} = \delta_1^{(2)}, \quad
\frac{\partial L}{\partial z_2^{(2)}} = \delta_2^{(2)},
$$
$$
\frac{\partial z_1^{(2)}}{\partial h_1} = w_{11}^{(2)}, \quad
\frac{\partial z_2^{(2)}}{\partial h_1} = w_{21}^{(2)}, \quad
\frac{\partial h_1}{\partial z_1^{(1)}} = \sigma'(z_1^{(1)}).
$$
Получаем  
$$
\delta_1^{(1)} = \left( \delta_1^{(2)} w_{11}^{(2)} + \delta_2^{(2)} w_{21}^{(2)} \right) \sigma'(z_1^{(1)}).
$$
Заметим, что выражение в скобках есть не что иное, как скалярное произведение вектора $\boldsymbol\delta^{(2)}$ на первый столбец матрицы $\mathbf W^{(2)}$. Это соответствует умножению $\mathbf W^{(2)\top}$ на $\boldsymbol\delta^{(2)}$.

Аналогично для второго скрытого нейрона:  
$$
\delta_2^{(1)} = \left( \delta_1^{(2)} w_{12}^{(2)} + \delta_2^{(2)} w_{22}^{(2)} \right) \sigma'(z_2^{(1)}).
$$
В матричной форме оба выражения объединяются:  
$$
\boldsymbol\delta^{(1)} = \left( \mathbf W^{(2)\top} \boldsymbol\delta^{(2)} \right) \odot \sigma'(\mathbf z^{(1)}),
$$
где $\odot$ — поэлементное произведение.

**Шаг 6. Градиенты параметров скрытого слоя.**  
Имея $\delta_1^{(1)}$ и $\delta_2^{(1)}$, находим градиенты по весам и смещениям скрытого слоя. Для каждого $j=1,2$ (номер скрытого нейрона) и $k=1,2$ (номер входа):  
$$
\frac{\partial L}{\partial w_{jk}^{(1)}} = \delta_j^{(1)} \cdot x_k, \qquad
\frac{\partial L}{\partial b_j^{(1)}} = \delta_j^{(1)}.
$$
В матричной форме:  
$$
\frac{\partial L}{\partial \mathbf W^{(1)}} = \boldsymbol\delta^{(1)} \mathbf x^\top, \qquad
\frac{\partial L}{\partial \mathbf b^{(1)}} = \boldsymbol\delta^{(1)}.
$$

#### 5.5. Обновление параметров

Все параметры обновляются по правилу градиентного спуска:  
$$
\mathbf W^{(\ell)} := \mathbf W^{(\ell)} - \eta \frac{\partial L}{\partial \mathbf W^{(\ell)}}, \qquad
\mathbf b^{(\ell)} := \mathbf b^{(\ell)} - \eta \frac{\partial L}{\partial \mathbf b^{(\ell)}}, \quad \ell=1,2.
$$

---

### 6. Обобщение и значение $\delta$-нотации

Проведённые выкладки для трёх частных случаев демонстрируют общий принцип алгоритма обратного распространения ошибки. Введённые локальные градиенты $\delta_i^{(\ell)}$ позволяют единообразно записать вычисления для любого слоя сети.

Для слоя $\ell$ (не выходного) справедлива рекуррентная формула:  
$$
\boldsymbol\delta^{(\ell)} = \left( \mathbf W^{(\ell+1)\top} \boldsymbol\delta^{(\ell+1)} \right) \odot \sigma'(\mathbf z^{(\ell)}).
$$
Для выходного слоя $L$:  
$$
\delta_i^{(L)} = \frac{\partial L}{\partial \hat y_i} \cdot \sigma'(z_i^{(L)}).
$$
После того как все $\boldsymbol\delta^{(\ell)}$ вычислены, градиенты по параметрам находятся как  
$$
\frac{\partial L}{\partial \mathbf W^{(\ell)}} = \boldsymbol\delta^{(\ell)} \left( \mathbf a^{(\ell-1)} \right)^\top, \qquad
\frac{\partial L}{\partial \mathbf b^{(\ell)}} = \boldsymbol\delta^{(\ell)}.
$$
Эти формулы составляют ядро алгоритма обратного распространения и непосредственно переносятся на сети произвольной глубины и ширины. Введение $\delta$ не только сокращает запись, но и выявляет структурную симметрию, позволяя реализовать алгоритм в матричном виде, что критично для эффективных вычислений.

Таким образом, мы подробно рассмотрели три примера MLP, вывели все необходимые формулы и объяснили происхождение сумм в обратном распространении. Полученные результаты являются фундаментом для понимания более сложных архитектур нейронных сетей.